# Analytical source annotations for a literature review

This notebook processes two source layouts:

1. A TXT file in the input directory root is treated as a standalone article or chapter.
2. Each first-level folder is treated as one composite source; all TXT files below it are its fragments.

The Excel output contains one row per source, section mappings, fragment notes, verified verbatim definitions, and a processing log.

Before running, set INPUT_DIR and review OUTPUT_DIR and OUTPUT_LANGUAGE. RUN_ANALYSIS is disabled by default for safety.




In [ ]:
# If required packages are missing, uncomment and run the next line once:
# %pip install -q -U openai pandas openpyxl

from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from getpass import getpass
from pathlib import Path
from typing import Any

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None





## 1. Configuration

The input and output directories must be separate. Place each composite source in its own first-level folder.




In [ ]:
INPUT_DIR = Path("/Users/ikudinov/Documents/Lit_Cinema")
OUTPUT_DIR = Path("/Users/ikudinov/Documents/Lit_Cinema_output")
RESULT_XLSX = OUTPUT_DIR / "literature_review_annotations.xlsx"
CACHE_DIR = OUTPUT_DIR / "_cache"

# Recommended balance: a lower-cost model for fragment extraction and a stronger model for final synthesis.
CHUNK_MODEL = "gpt-5.4-mini"
SYNTHESIS_MODEL = "gpt-5.4"

# Language used for analytical descriptions. Verbatim definition quotes always remain in the source language.
OUTPUT_LANGUAGE = "English"

# OPENAI_API_KEY takes precedence. Otherwise, the notebook reads the key from this local file.
# The key itself is never written to the notebook or the Excel output.
API_KEY_PATH: Path | None = Path("/Users/ikudinov/Documents/keys/api_openai_ab.txt")

REQUEST_TIMEOUT_SECONDS = 300
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 4
SLEEP_BETWEEN_REQUESTS = 0.4

# Long standalone files automatically switch to fragment mode instead of being skipped or truncated.
DIRECT_SOURCE_MAX_CHARS = 80_000
CHUNK_SIZE_CHARS = 45_000
CHUNK_OVERLAP_CHARS = 2_500
MAX_SYNTHESIS_INPUT_CHARS = 180_000
REDUCTION_BATCH_CHARS = 70_000
MAX_DEFINITION_QUOTE_CHARS = 2_000

SKIP_CACHED = True
CHECKPOINT_EVERY_SOURCES = 25
RUN_ANALYSIS = True

SUPPORTED_EXTENSIONS = {".txt"}





## 2. Research framework and dissertation sections

The framework extends beyond film analysis. It includes audience reception, the character as a representation of life agency, naturally occurring digital traces, and methods for analysing existing texts and transcripts.




In [ ]:
RESEARCH_TOPIC = (
    "The social representation of the film hero in mass consciousness: "
    "sociocultural narratives, audience reception, and digital methods of analysis"
)

RESEARCH_OBJECT = "the social representation of the film hero in mass consciousness"

RESEARCH_SUBJECT = (
    "sociocultural narratives in which the film hero is represented as a bearer of agency, "
    "collective identity, value orientations, and symbolic models of action"
)

RESEARCH_AXES = {
    "A": "cinema, cinematic texts, visual culture, and approaches to film analysis",
    "B": "film characters and film heroes: functions, typologies, and narrative or symbolic construction",
    "C": "life agency, subjectivity, autonomy, responsibility, identity, and models of action",
    "D": "mass consciousness, collective representations, audience reception, interpretation, and identification",
    "E": "reviews, criticism, comments, forums, social media, video platforms, and transcripts",
    "F": "sociological and digital methods: content analysis, NLP, ML, topic modelling, emotion analysis, network analysis, and mixed methods",
    "G": "sociocultural transformation, war, memory, official and alternative narratives, and local or global identity",
}

SECTION_MAP = {
    "1.1": "Ontology and semiotics of the film hero; functions, archetypes, and collective memory",
    "1.2": "Mass consciousness; structure, reception mechanisms, and narrative dynamics",
    "1.3": "The sociocultural narrative as an instrument for constructing the heroic",
    "1.4": "State of research in the sociology of culture, visual studies, and media studies",
    "1.5": "Approaches and principles for studying the film hero in mass consciousness",
    "2.1": "History of sociological analysis of visual and narrative images",
    "2.2": "Methods of analysing cinematic texts, from content analysis to digital technologies",
    "2.3": "NLP and ML in the study of audience reception",
    "2.4": "Hybrid analysis integrating qualitative and quantitative methods",
    "2.5": "Benefits, risks, and ethical limits of AI in cultural analysis",
    "3.1": "Types of heroic narratives: hero, antihero, post-hero, and traumatic hero",
    "3.2": "Political, moral, ideological, and technological determinants of hero transformation",
    "3.3": "Interaction between elite and mass culture in representations of the heroic",
    "3.4": "Influence of media environments and digital platforms on the reinterpretation of the hero",
    "3.5": "The hero as a social construct in fan culture, memes, and online discussions",
    "4.1": "Building a corpus of screenplays, reviews, criticism, comments, and video transcripts",
    "4.2": "Automated content analysis, semantic mapping, and narrative patterns",
    "4.3": "Comparison of Western and Eastern film heroes",
    "4.4": "The Ukrainian context: Soviet, post-Soviet, and wartime periods",
    "4.5": "Reconstructing changes in heroic archetypes in mass consciousness",
    "5.1": "The hero between official and alternative narratives",
    "5.2": "Media standardisation and threats to local identities",
    "5.3": "New forms of the heroic in the digital age",
    "5.4": "A model of future transformations of the heroic",
    "5.5": "Practical recommendations for filmmakers, institutions, and sociologists",
}

VALID_SECTION_CODES = list(SECTION_MAP)





## 3. Structured output

The model returns schema-constrained JSON. This makes the Excel export reliable and keeps relevance scores separate from free-form prose.




In [ ]:
DEFINITION_ITEM_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "concept": {"type": "string"},
        "definition_quote": {"type": "string"},
        "source_language": {"type": "string", "enum": ["Ukrainian", "English"]},
        "definition_type": {
            "type": "string",
            "enum": ["explicit_definition", "operational_definition"],
        },
        "location_hint": {"type": "string"},
    },
    "required": [
        "concept",
        "definition_quote",
        "source_language",
        "definition_type",
        "location_hint",
    ],
    "additionalProperties": False,
}


PARTIAL_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "fragment_summary": {"type": "string"},
        "relevance_to_study": {"type": "string"},
        "contributions": {"type": "array", "items": {"type": "string"}},
        "key_concepts": {"type": "array", "items": {"type": "string"}},
        "definitions": {"type": "array", "items": DEFINITION_ITEM_SCHEMA},
        "methods_and_data": {"type": "array", "items": {"type": "string"}},
        "character_or_subject_material": {"type": "array", "items": {"type": "string"}},
        "reception_or_platform_material": {"type": "array", "items": {"type": "string"}},
        "limitations": {"type": "array", "items": {"type": "string"}},
        "candidate_sections": {
            "type": "array",
            "items": {"type": "string", "enum": VALID_SECTION_CODES},
        },
        "evidence_anchors": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "fragment_summary",
        "relevance_to_study",
        "contributions",
        "key_concepts",
        "definitions",
        "methods_and_data",
        "character_or_subject_material",
        "reception_or_platform_material",
        "limitations",
        "candidate_sections",
        "evidence_anchors",
    ],
    "additionalProperties": False,
}

FINAL_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "annotation": {"type": "string"},
        "object_and_subject": {"type": "string"},
        "problem_and_argument": {"type": "string"},
        "theoretical_framework": {"type": "string"},
        "key_concepts": {"type": "array", "items": {"type": "string"}},
        "definitions": {"type": "array", "items": DEFINITION_ITEM_SCHEMA},
        "methods_and_data": {"type": "string"},
        "main_results_or_contribution": {"type": "string"},
        "state_of_research_value": {"type": "string"},
        "research_gap_or_unresolved_issue": {"type": "string"},
        "cinema_relevance": {"type": "string"},
        "character_subject_relevance": {"type": "string"},
        "mass_consciousness_relevance": {"type": "string"},
        "digital_methods_relevance": {"type": "string"},
        "limitations": {"type": "string"},
        "overall_relevance_score": {"type": "integer", "minimum": 0, "maximum": 4},
        "overall_relevance_label": {
            "type": "string",
            "enum": ["core", "high", "medium", "low", "none"],
        },
        "primary_cluster": {
            "type": "string",
            "enum": [
                "cinema_and_film_analysis",
                "film_character",
                "subjectivity_and_identity",
                "mass_consciousness_and_reception",
                "digital_platforms_and_traces",
                "research_methods",
                "sociocultural_context",
                "unrelated",
            ],
        },
        "relevance_rationale": {"type": "string"},
        "source_roles": {"type": "array", "items": {"type": "string"}},
        "section_matches": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "section_code": {"type": "string", "enum": VALID_SECTION_CODES},
                    "fit_score": {"type": "integer", "minimum": 1, "maximum": 3},
                    "rationale": {"type": "string"},
                },
                "required": ["section_code", "fit_score", "rationale"],
                "additionalProperties": False,
            },
        },
        "recommended_use": {"type": "string"},
        "exclusion_reason": {"type": "string"},
        "source_completeness": {
            "type": "string",
            "enum": ["complete", "probably_complete", "fragmentary", "unclear"],
        },
        "confidence": {"type": "integer", "minimum": 1, "maximum": 3},
        "keywords": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "annotation",
        "object_and_subject",
        "problem_and_argument",
        "theoretical_framework",
        "key_concepts",
        "definitions",
        "methods_and_data",
        "main_results_or_contribution",
        "state_of_research_value",
        "research_gap_or_unresolved_issue",
        "cinema_relevance",
        "character_subject_relevance",
        "mass_consciousness_relevance",
        "digital_methods_relevance",
        "limitations",
        "overall_relevance_score",
        "overall_relevance_label",
        "primary_cluster",
        "relevance_rationale",
        "source_roles",
        "section_matches",
        "recommended_use",
        "exclusion_reason",
        "source_completeness",
        "confidence",
        "keywords",
    ],
    "additionalProperties": False,
}





## 4. Analytical instructions




In [ ]:
def research_context() -> str:
    axes = "\n".join(f"{code}. {text}" for code, text in RESEARCH_AXES.items())
    sections = "\n".join(f"{code}: {text}" for code, text in SECTION_MAP.items())
    return f"""
TOPIC: {RESEARCH_TOPIC}
OBJECT OF STUDY: {RESEARCH_OBJECT}
SUBJECT OF STUDY: {RESEARCH_SUBJECT}

ANALYTICAL AXES:
{axes}

DISSERTATION SECTIONS:
{sections}
""".strip()


COMMON_RULES = f"""
You are a research assistant specialising in the sociology of culture, visual and media studies,
theories of subjectivity and agency, and digital methods of text analysis.

Write all analytical descriptions in {OUTPUT_LANGUAGE}. Values that function as codes or labels
must follow the English values defined by the JSON schema. Verbatim definition quotes are the only
exception: preserve them exactly in the original Ukrainian or English.

Critical rules:
1. Analyse only the supplied text. Do not add external facts or invent bibliographic information.
2. Do not identify the author or authors by given name or surname in analytical fields. Use only
   the equivalents of “the author”, “the authors”, or “the researcher”, or use impersonal wording.
3. Do not replace the source's actual content with the dissertation topic. Clearly distinguish:
   a) what the source actually studies; b) how it can be used; c) what is absent from the source.
4. A source may be valuable through only one analytical axis. Do not lower its relevance merely
   because it does not discuss cinema if it makes a substantial contribution to agency, identity,
   mass consciousness, reception, or a relevant research method.
5. Do not manufacture connections. If a source does not address any analytical axis, set
   score=0, label=none, cluster=unrelated, and provide a short exclusion reason.
6. Distinguish representation of a character in a film from audience perception of that character.
   Claims about mass consciousness require reception data or an explicit argument about reception.
7. Identify studies of naturally occurring digital traces: reviews, criticism, comments,
   discussions, videos, and transcripts. Do not treat these data as surveys.
8. For methodological sources, record data type, unit of analysis, procedure, validation,
   limitations, ethical risks, and reproducibility when the text provides them.
9. If the text is fragmentary or damaged by OCR, reduce confidence and state the limits explicitly.
10. Avoid unsupported evaluative phrases such as “an important study”. State the concrete contribution.
11. Extract a definition only when the source contains an explicit definitional or operational
    formulation. Never turn your interpretation of a concept into a definition.
12. Every definition_quote must be copied verbatim from the supplied source text. Preserve the
    original language, wording, spelling, punctuation, and grammatical form. Do not translate,
    paraphrase, correct, combine non-contiguous passages, or silently insert ellipses.
13. Select the shortest self-contained passage that preserves the complete definition, normally
    one to three sentences and never more than {MAX_DEFINITION_QUOTE_CHARS} characters.
14. If no exact definitional passage is present, return an empty definitions array. A concept's use,
    discussion, examples, attributes, or your inferred understanding do not count as a definition.
""".strip()


PARTIAL_INSTRUCTION = COMMON_RULES + "\n\n" + """
Prepare a compact, evidence-based note for one fragment of a larger source.
Do not characterise the entire work. Extract only what this fragment explicitly supports.
evidence_anchors should be short verification aids such as a section heading, subsection heading,
table label, or recognisable content marker without a long quotation.

For definitions, concept is a concise concept label, while definition_quote is strictly a verbatim
quotation. location_hint is a heading or another short locator; it must not replace the quote.
""" + "\n\n" + research_context()


REDUCTION_INSTRUCTION = COMMON_RULES + "\n\n" + """
Compress a set of partial notes into one consolidated partial note.
Do not introduce new conclusions. Preserve contradictions, limitations, methods, key concepts,
and the strongest evidence anchors. Copy any retained definition_quote without changing a character.
This is an intermediate reduction step, not the final annotation.
""" + "\n\n" + research_context()


FINAL_INSTRUCTION = COMMON_RULES + "\n\n" + """
Prepare a final analytical source record for writing the state-of-research section.
The annotation should not retell the text sequentially. Reconstruct the research problem,
argumentative logic, conceptual apparatus, methods, results, contribution to the field,
and unresolved questions.

Required annotation length:
- core/high: approximately 700–1,100 words in 6–9 substantive paragraphs;
- medium: 450–750 words in 4–7 paragraphs;
- low: 250–450 words;
- none: 120–250 words with a neutral description of the actual topic and exclusion reason.
Do not pad the annotation when the source is fragmentary or lacks relevant evidence.

Relevance scale:
4/core: directly supports a key theoretical or methodological part of the study;
3/high: makes a substantial contribution to at least one analytical axis;
2/medium: useful as context, comparison, or partial methodological support;
1/low: peripheral connection suitable only for a targeted citation;
0/none: no substantive connection.

section_matches must include only genuinely supported sections. fit_score 3 means a direct
contribution, 2 a substantial supporting contribution, and 1 contextual use. For an unrelated
source, return an empty array. Leave exclusion_reason empty unless the label is none.

When the full source text is supplied, extract verified candidate definitions in definitions.
When the input consists of partial notes, preserve only verbatim quotes already present in them;
never reconstruct a quote from summaries.
""" + "\n\n" + research_context()





## 5. Source discovery and text reading




In [ ]:
@dataclass(frozen=True)
class SourceItem:
    source_id: str
    source_type: str
    root_path: Path
    files: tuple[Path, ...]


def natural_key(path: Path) -> list[Any]:
    parts = re.split(r"(\d+)", path.as_posix().casefold())
    return [int(part) if part.isdigit() else part for part in parts]


def discover_sources(input_dir: Path) -> list[SourceItem]:
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    if not input_dir.is_dir():
        raise NotADirectoryError(f"Expected a directory: {input_dir}")

    root_files = sorted(
        (
            p for p in input_dir.iterdir()
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS and not p.name.startswith(".")
        ),
        key=natural_key,
    )
    root_dirs = sorted(
        (p for p in input_dir.iterdir() if p.is_dir() and not p.name.startswith(".")),
        key=natural_key,
    )

    sources = [
        SourceItem(p.stem, "standalone", p, (p,))
        for p in root_files
    ]

    for folder in root_dirs:
        files = tuple(sorted(
            (
                p for p in folder.rglob("*")
                if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS and not p.name.startswith(".")
            ),
            key=natural_key,
        ))
        if files:
            sources.append(SourceItem(folder.name, "composite", folder, files))

    source_ids = [source.source_id for source in sources]
    duplicates = sorted({item for item in source_ids if source_ids.count(item) > 1})
    if duplicates:
        raise ValueError(
            "Duplicate first-level file and folder identifiers: "
            + ", ".join(duplicates)
            + ". Rename them so that each unit of analysis has a unique identifier."
        )

    return sources


def read_text_file(path: Path) -> tuple[str, str]:
    encodings = ("utf-8", "utf-8-sig", "cp1251", "windows-1252", "latin-1")
    for encoding in encodings:
        try:
            return path.read_text(encoding=encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Could not decode: {path}")


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{4,}", "\n\n\n", text)
    return text.strip()


def normalise_whitespace_for_quote_check(text: str) -> str:
    text = unicodedata.normalize("NFC", text).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


def verify_definition_quote(quote: str, source_text: str) -> str:
    if not quote.strip():
        return "rejected_empty"
    if len(quote) > MAX_DEFINITION_QUOTE_CHARS:
        return "rejected_too_long"
    if quote in source_text:
        return "exact_match"
    normalised_quote = normalise_whitespace_for_quote_check(quote)
    normalised_source = normalise_whitespace_for_quote_check(source_text)
    if normalised_quote and normalised_quote in normalised_source:
        return "whitespace_normalised_match"
    return "rejected_not_found"


def verified_definition_items(
    candidates: list[dict[str, Any]],
    *,
    source: SourceItem,
    fragment_file: str,
    chunk_index: int,
    total_chunks: int,
    source_text: str,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    verified_items: list[dict[str, Any]] = []
    export_rows: list[dict[str, Any]] = []
    rejected_count = 0

    for candidate in candidates:
        quote = str(candidate.get("definition_quote", ""))
        status = verify_definition_quote(quote, source_text)
        if status.startswith("rejected_"):
            rejected_count += 1
            continue

        item = {
            "concept": str(candidate.get("concept", "")),
            "definition_quote": quote,
            "source_language": str(candidate.get("source_language", "")),
            "definition_type": str(candidate.get("definition_type", "")),
            "location_hint": str(candidate.get("location_hint", "")),
        }
        verified_items.append(item)
        export_rows.append({
            "source_id": source.source_id,
            "source_type": source.source_type,
            "fragment_file": fragment_file,
            "chunk_index": chunk_index,
            "total_chunks": total_chunks,
            **item,
            "quote_characters": len(quote),
            "verification_status": status,
            "quote_verified": True,
        })

    return verified_items, export_rows, rejected_count


def deduplicate_definition_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    unique: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for row in rows:
        key = (
            row["source_id"],
            normalise_whitespace_for_quote_check(str(row["definition_quote"])),
        )
        if key not in seen:
            seen.add(key)
            unique.append(row)
    return unique


def split_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("CHUNK_SIZE_CHARS must be greater than CHUNK_OVERLAP_CHARS")
    if len(text) <= chunk_size:
        return [text]

    chunks: list[str] = []
    start = 0
    while start < len(text):
        proposed_end = min(start + chunk_size, len(text))
        end = proposed_end
        if proposed_end < len(text):
            search_start = max(start + chunk_size // 2, proposed_end - 4_000)
            paragraph_break = text.rfind("\n\n", search_start, proposed_end)
            sentence_break = max(
                text.rfind(". ", search_start, proposed_end),
                text.rfind("! ", search_start, proposed_end),
                text.rfind("? ", search_start, proposed_end),
            )
            if paragraph_break > start:
                end = paragraph_break
            elif sentence_break > start:
                end = sentence_break + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


def source_fingerprint(source: SourceItem, input_dir: Path) -> str:
    digest = hashlib.sha256()
    digest.update(source.source_type.encode("utf-8"))
    digest.update(PARTIAL_INSTRUCTION.encode("utf-8"))
    digest.update(REDUCTION_INSTRUCTION.encode("utf-8"))
    digest.update(FINAL_INSTRUCTION.encode("utf-8"))
    for path in source.files:
        digest.update(path.relative_to(input_dir).as_posix().encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def safe_slug(value: str) -> str:
    slug = re.sub(r"[^\w.-]+", "_", value, flags=re.UNICODE).strip("._")
    return (slug[:100] or "source")





## 6. API calls, retries, and cache




In [ ]:
API_CALL_LOG: list[dict[str, Any]] = []


def load_api_key() -> str:
    env_key = os.getenv("OPENAI_API_KEY", "").strip()
    if env_key:
        return env_key
    if API_KEY_PATH is not None:
        key_path = Path(API_KEY_PATH)
        if not key_path.exists():
            raise FileNotFoundError(f"API key file not found: {key_path}")
        file_key = key_path.read_text(encoding="utf-8").strip()
        if file_key:
            return file_key
    entered = getpass("Enter OPENAI_API_KEY (input is hidden): ").strip()
    if not entered:
        raise ValueError("No API key was provided")
    return entered


def call_structured(
    client: OpenAI,
    *,
    model: str,
    instructions: str,
    input_text: str,
    schema: dict[str, Any],
    schema_name: str,
    max_output_tokens: int,
    stage: str,
    source_id: str,
) -> dict[str, Any]:
    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        started = time.time()
        try:
            response = client.responses.create(
                model=model,
                instructions=instructions,
                input=input_text,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "strict": True,
                        "schema": schema,
                    }
                },
                max_output_tokens=max_output_tokens,
                store=False,
            )
            if not response.output_text:
                raise ValueError("The model returned an empty response")
            parsed = json.loads(response.output_text)
            usage = getattr(response, "usage", None)
            API_CALL_LOG.append({
                "source_id": source_id,
                "stage": stage,
                "model": model,
                "attempt": attempt,
                "status": "ok",
                "elapsed_seconds": round(time.time() - started, 2),
                "input_tokens": getattr(usage, "input_tokens", None),
                "output_tokens": getattr(usage, "output_tokens", None),
                "response_id": getattr(response, "id", ""),
                "error": "",
            })
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            return parsed
        except Exception as exc:
            last_error = exc
            API_CALL_LOG.append({
                "source_id": source_id,
                "stage": stage,
                "model": model,
                "attempt": attempt,
                "status": "failed_attempt",
                "elapsed_seconds": round(time.time() - started, 2),
                "input_tokens": None,
                "output_tokens": None,
                "response_id": "",
                "error": str(exc),
            })
            if attempt < MAX_RETRIES:
                delay = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
                print(f"  API error; retrying in {delay:.1f} seconds: {exc}")
                time.sleep(delay)
    raise RuntimeError(f"API call failed after {MAX_RETRIES} attempts: {last_error}")


def load_json_if_exists(path: Path) -> dict[str, Any] | None:
    if SKIP_CACHED and path.exists() and path.stat().st_size > 0:
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def save_json(path: Path, data: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")





## 7. Fragment analysis and hierarchical synthesis




In [ ]:
def analyze_chunk(
    client: OpenAI,
    *,
    source: SourceItem,
    file_path: Path,
    relative_name: str,
    chunk_text: str,
    chunk_index: int,
    total_chunks: int,
    source_hash: str,
) -> dict[str, Any]:
    cache_path = (
        CACHE_DIR / safe_slug(source.source_id) /
        f"{source_hash[:16]}_{safe_slug(relative_name)}_{chunk_index:04d}.json"
    )
    cached = load_json_if_exists(cache_path)
    if cached is not None:
        return cached

    prompt = f"""
SOURCE: a unit of a composite work; do not identify the author by name.
FRAGMENT FILE: {relative_name}
PART: {chunk_index} of {total_chunks}

SOURCE FRAGMENT:
{chunk_text}
""".strip()

    result = call_structured(
        client,
        model=CHUNK_MODEL,
        instructions=PARTIAL_INSTRUCTION,
        input_text=prompt,
        schema=PARTIAL_SCHEMA,
        schema_name="fragment_note",
        max_output_tokens=2_200,
        stage="fragment",
        source_id=source.source_id,
    )
    save_json(cache_path, result)
    return result


def grouped_by_size(items: list[dict[str, Any]], max_chars: int) -> list[list[dict[str, Any]]]:
    groups: list[list[dict[str, Any]]] = []
    current: list[dict[str, Any]] = []
    current_size = 0
    for item in items:
        item_size = len(json.dumps(item, ensure_ascii=False))
        if current and current_size + item_size > max_chars:
            groups.append(current)
            current = []
            current_size = 0
        current.append(item)
        current_size += item_size
    if current:
        groups.append(current)
    return groups


def reduce_notes_if_needed(
    client: OpenAI,
    source_id: str,
    notes: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    round_no = 0
    reduced = notes
    while len(json.dumps(reduced, ensure_ascii=False)) > MAX_SYNTHESIS_INPUT_CHARS:
        round_no += 1
        batches = grouped_by_size(reduced, REDUCTION_BATCH_CHARS)
        next_round: list[dict[str, Any]] = []
        print(f"  Intermediate reduction: round {round_no}, batches {len(batches)}")
        for batch_no, batch in enumerate(batches, start=1):
            result = call_structured(
                client,
                model=CHUNK_MODEL,
                instructions=REDUCTION_INSTRUCTION,
                input_text=json.dumps(batch, ensure_ascii=False),
                schema=PARTIAL_SCHEMA,
                schema_name="reduced_fragment_notes",
                max_output_tokens=2_500,
                stage=f"reduction_{round_no}_{batch_no}",
                source_id=source_id,
            )
            next_round.append(result)
        if len(next_round) >= len(reduced):
            raise RuntimeError("Intermediate reduction did not reduce the number of notes")
        reduced = next_round
    return reduced


def synthesize_final(
    client: OpenAI,
    *,
    source: SourceItem,
    source_hash: str,
    content_payload: str,
    mode: str,
) -> dict[str, Any]:
    cache_path = CACHE_DIR / safe_slug(source.source_id) / f"{source_hash[:16]}_FINAL.json"
    cached = load_json_if_exists(cache_path)
    if cached is not None:
        return cached

    prompt = f"""
SOURCE TYPE: {source.source_type}
INPUT MODE: {mode}
NUMBER OF FILES: {len(source.files)}

INPUT MATERIAL:
{content_payload}
""".strip()

    result = call_structured(
        client,
        model=SYNTHESIS_MODEL,
        instructions=FINAL_INSTRUCTION,
        input_text=prompt,
        schema=FINAL_SCHEMA,
        schema_name="source_analytical_card",
        max_output_tokens=7_500,
        stage="final_synthesis",
        source_id=source.source_id,
    )
    save_json(cache_path, result)
    return result





## 8. Processing one source




In [ ]:
def process_source(
    client: OpenAI,
    source: SourceItem,
    input_dir: Path,
) -> tuple[dict[str, Any], list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    source_hash = source_fingerprint(source, input_dir)
    total_chars = 0
    encodings: set[str] = set()
    prepared_files: list[tuple[Path, str, str]] = []

    for path in source.files:
        raw, encoding = read_text_file(path)
        text = clean_text(raw)
        if not text:
            continue
        relative_name = path.relative_to(source.root_path if source.source_type == "composite" else input_dir).as_posix()
        prepared_files.append((path, relative_name, text))
        total_chars += len(text)
        encodings.add(encoding)

    if not prepared_files:
        raise ValueError("All source files are empty after cleaning")

    partial_rows: list[dict[str, Any]] = []
    definition_rows: list[dict[str, Any]] = []

    # A short standalone source is analysed in a single final call.
    if source.source_type == "standalone" and total_chars <= DIRECT_SOURCE_MAX_CHARS:
        content_payload = prepared_files[0][2]
        final = synthesize_final(
            client,
            source=source,
            source_hash=source_hash,
            content_payload=content_payload,
            mode="full text of a standalone source",
        )
        verified_items, verified_rows, rejected_count = verified_definition_items(
            final.get("definitions", []),
            source=source,
            fragment_file=prepared_files[0][1],
            chunk_index=1,
            total_chunks=1,
            source_text=prepared_files[0][2],
        )
        final["definitions"] = verified_items
        definition_rows.extend(verified_rows)
        if rejected_count:
            print(f"  Rejected unverified definition quotes: {rejected_count}")
        processing_mode = "direct"
    else:
        notes_for_synthesis: list[dict[str, Any]] = []
        for file_no, (path, relative_name, text) in enumerate(prepared_files, start=1):
            chunks = split_text(text, CHUNK_SIZE_CHARS, CHUNK_OVERLAP_CHARS)
            print(f"  File {file_no}/{len(prepared_files)}: {relative_name}; chunks: {len(chunks)}")
            for chunk_index, chunk_text in enumerate(chunks, start=1):
                note = analyze_chunk(
                    client,
                    source=source,
                    file_path=path,
                    relative_name=relative_name,
                    chunk_text=chunk_text,
                    chunk_index=chunk_index,
                    total_chunks=len(chunks),
                    source_hash=source_hash,
                )
                verified_items, verified_rows, rejected_count = verified_definition_items(
                    note.get("definitions", []),
                    source=source,
                    fragment_file=relative_name,
                    chunk_index=chunk_index,
                    total_chunks=len(chunks),
                    source_text=chunk_text,
                )
                note["definitions"] = verified_items
                definition_rows.extend(verified_rows)
                note_with_context = {
                    "fragment_file": relative_name,
                    "chunk_index": chunk_index,
                    "total_chunks": len(chunks),
                    "rejected_definition_quotes": rejected_count,
                    **note,
                }
                notes_for_synthesis.append(note_with_context)
                partial_rows.append({
                    "source_id": source.source_id,
                    "source_type": source.source_type,
                    **note_with_context,
                })

        reduced_notes = reduce_notes_if_needed(client, source.source_id, notes_for_synthesis)
        content_payload = json.dumps(reduced_notes, ensure_ascii=False, indent=2)
        final = synthesize_final(
            client,
            source=source,
            source_hash=source_hash,
            content_payload=content_payload,
            mode="synthesis of evidence-based notes from fragments of one source",
        )
        processing_mode = "fragment_synthesis"

    section_codes = [m["section_code"] for m in final.get("section_matches", [])]
    source_path = source.root_path.relative_to(input_dir).as_posix()
    row = {
        "source_id": source.source_id,
        "source_type": source.source_type,
        "source_path": source_path,
        "file_count": len(prepared_files),
        "input_characters": total_chars,
        "encodings": " | ".join(sorted(encodings)),
        "processing_mode": processing_mode,
        "source_fingerprint": source_hash,
        "processed_at": datetime.now().isoformat(timespec="seconds"),
        **{k: v for k, v in final.items() if k not in {"section_matches", "definitions"}},
        "matched_sections": " | ".join(section_codes),
    }
    return row, partial_rows, deduplicate_definition_rows(definition_rows), final


def flatten_value(value: Any) -> Any:
    if isinstance(value, list):
        return " | ".join(str(v) for v in value)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return value





## 9. Excel export




In [ ]:
EXCEL_CELL_LIMIT = 32_000
EXCEL_ILLEGAL_CHARACTERS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")


def excel_safe(value: Any) -> Any:
    value = flatten_value(value)
    if isinstance(value, str):
        # XLSX uses XML, which forbids these invisible C0 control characters.
        # Replace them with spaces so neighbouring words do not become concatenated.
        value = EXCEL_ILLEGAL_CHARACTERS_RE.sub(" ", value)
    if isinstance(value, str) and len(value) > EXCEL_CELL_LIMIT:
        return value[: EXCEL_CELL_LIMIT - 60] + " … [truncated because of the Excel cell limit]"
    return value


def make_section_rows(source_rows: list[dict[str, Any]], finals_by_id: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for source_row in source_rows:
        source_id = source_row["source_id"]
        for match in finals_by_id[source_id].get("section_matches", []):
            rows.append({
                "source_id": source_id,
                "section_code": match["section_code"],
                "section_title": SECTION_MAP[match["section_code"]],
                "fit_score": match["fit_score"],
                "rationale": match["rationale"],
            })
    return rows


def style_worksheet(ws, widths: dict[str, int] | None = None) -> None:
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    header_fill = PatternFill("solid", fgColor="1F4E78")
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 34

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    for index, column_cells in enumerate(ws.columns, start=1):
        header = str(column_cells[0].value or "")
        if widths and header in widths:
            width = widths[header]
        else:
            sample = [str(c.value or "") for c in column_cells[:80]]
            width = min(max(max((len(v) for v in sample), default=8) + 2, 10), 42)
        ws.column_dimensions[get_column_letter(index)].width = width


def export_excel(
    source_rows: list[dict[str, Any]],
    section_rows: list[dict[str, Any]],
    definition_rows: list[dict[str, Any]],
    fragment_rows: list[dict[str, Any]],
    run_rows: list[dict[str, Any]],
) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    source_columns = [
        "source_id", "source_type", "source_path", "file_count", "input_characters",
        "processing_mode", "overall_relevance_score", "overall_relevance_label",
        "primary_cluster", "confidence", "source_completeness", "matched_sections",
        "source_roles", "keywords", "annotation", "object_and_subject",
        "problem_and_argument", "theoretical_framework", "methods_and_data",
        "main_results_or_contribution", "state_of_research_value",
        "research_gap_or_unresolved_issue", "cinema_relevance",
        "character_subject_relevance", "mass_consciousness_relevance",
        "digital_methods_relevance", "limitations", "relevance_rationale",
        "recommended_use", "exclusion_reason", "encodings", "source_fingerprint",
        "processed_at",
    ]
    sources_df = pd.DataFrame(source_rows).reindex(columns=source_columns)
    sections_df = pd.DataFrame(section_rows, columns=[
        "source_id", "section_code", "section_title", "fit_score", "rationale"
    ])
    definitions_df = pd.DataFrame(definition_rows, columns=[
        "source_id", "source_type", "fragment_file", "chunk_index", "total_chunks",
        "concept", "definition_quote", "source_language", "definition_type",
        "location_hint", "quote_characters", "verification_status", "quote_verified",
    ])
    fragments_df = pd.DataFrame(fragment_rows)
    run_df = pd.DataFrame(run_rows)

    for df in (sources_df, sections_df, definitions_df, fragments_df, run_df):
        for column in df.columns:
            df[column] = df[column].map(excel_safe)

    readme_rows = [
        ["Parameter", "Value"],
        ["Research topic", RESEARCH_TOPIC],
        ["Object of study", RESEARCH_OBJECT],
        ["Subject of study", RESEARCH_SUBJECT],
        ["Input directory", str(INPUT_DIR)],
        ["Fragment model", CHUNK_MODEL],
        ["Synthesis model", SYNTHESIS_MODEL],
        ["Analytical output language", OUTPUT_LANGUAGE],
        ["Unit represented by a Sources row", "one standalone or composite source"],
        ["Relevance scale", "0 none; 1 low; 2 medium; 3 high; 4 core"],
        ["Author names", "not used in analytical fields; the path is retained separately for traceability"],
        ["Definition quotes", "verbatim Ukrainian or English passages verified against the supplied source text"],
        ["Rejected definition candidates", "not exported to Definitions when the quote cannot be found in the source fragment"],
    ]
    readme_df = pd.DataFrame(readme_rows[1:], columns=readme_rows[0])

    with pd.ExcelWriter(RESULT_XLSX, engine="openpyxl") as writer:
        sources_df.to_excel(writer, sheet_name="Sources", index=False)
        definitions_df.to_excel(writer, sheet_name="Definitions", index=False)
        sections_df.to_excel(writer, sheet_name="Section_mapping", index=False)
        fragments_df.to_excel(writer, sheet_name="Fragment_notes", index=False)
        run_df.to_excel(writer, sheet_name="Run_log", index=False)
        readme_df.to_excel(writer, sheet_name="README", index=False)

        style_worksheet(writer.book["Sources"], {
            "source_id": 28, "source_path": 38, "annotation": 85,
            "object_and_subject": 48, "problem_and_argument": 55,
            "theoretical_framework": 45, "methods_and_data": 48,
            "main_results_or_contribution": 55, "state_of_research_value": 55,
            "research_gap_or_unresolved_issue": 50, "limitations": 48,
            "relevance_rationale": 48, "recommended_use": 48,
        })
        style_worksheet(writer.book["Definitions"], {
            "source_id": 28, "fragment_file": 38, "concept": 28,
            "definition_quote": 90, "location_hint": 48,
            "verification_status": 30,
        })
        style_worksheet(writer.book["Section_mapping"], {
            "source_id": 28, "section_title": 65, "rationale": 65,
        })
        style_worksheet(writer.book["Fragment_notes"], {
            "source_id": 28, "fragment_file": 38, "fragment_summary": 65,
            "relevance_to_study": 55, "evidence_anchors": 55,
        })
        style_worksheet(writer.book["Run_log"], {"source_id": 28, "error": 70})
        style_worksheet(writer.book["README"], {"Parameter": 30, "Value": 110})





## 10. Preview the input structure

This cell does not call the API or write any files.




In [ ]:
def preview_sources(input_dir: Path) -> pd.DataFrame:
    sources = discover_sources(input_dir)
    rows = []
    for source in sources:
        rows.append({
            "source_id": source.source_id,
            "source_type": source.source_type,
            "file_count": len(source.files),
            "path": str(source.root_path),
        })
    return pd.DataFrame(rows)


if INPUT_DIR.exists():
    preview_df = preview_sources(INPUT_DIR)
    print(f"Sources found: {len(preview_df)}")
    display(preview_df)
else:
    print(f"Set INPUT_DIR to an existing directory. Current path not found: {INPUT_DIR}")





## 11. Full run

Set RUN_ANALYSIS = True in the configuration and run this cell. Processing continues after individual source failures. The cache prevents repeated API calls for completed fragments and final records when neither the source nor the instructions have changed.




In [ ]:
def run_pipeline() -> Path:
    if OpenAI is None:
        raise ImportError("The openai package is not installed. Run the package installation cell.")
    sources = discover_sources(INPUT_DIR)
    if not sources:
        raise ValueError(f"No TXT sources were found in {INPUT_DIR}")
    if OUTPUT_DIR.resolve() == INPUT_DIR.resolve() or INPUT_DIR.resolve() in OUTPUT_DIR.resolve().parents:
        raise ValueError("OUTPUT_DIR must be outside INPUT_DIR")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    client = OpenAI(api_key=load_api_key(), timeout=REQUEST_TIMEOUT_SECONDS)

    source_rows: list[dict[str, Any]] = []
    fragment_rows: list[dict[str, Any]] = []
    definition_rows: list[dict[str, Any]] = []
    run_rows: list[dict[str, Any]] = []
    finals_by_id: dict[str, dict[str, Any]] = {}

    for index, source in enumerate(sources, start=1):
        print(f"\n[{index}/{len(sources)}] {source.source_id} ({source.source_type})")
        started = time.time()
        try:
            row, partials, definitions, final = process_source(client, source, INPUT_DIR)
            source_rows.append(row)
            fragment_rows.extend(partials)
            definition_rows.extend(definitions)
            definition_rows = deduplicate_definition_rows(definition_rows)
            finals_by_id[source.source_id] = final
            run_rows.append({
                "source_id": source.source_id,
                "status": "ok",
                "elapsed_seconds": round(time.time() - started, 2),
                "error": "",
            })
            print(f"  Completed: relevance={row['overall_relevance_score']} ({row['overall_relevance_label']})")
        except Exception as exc:
            run_rows.append({
                "source_id": source.source_id,
                "status": "failed",
                "elapsed_seconds": round(time.time() - started, 2),
                "error": str(exc),
            })
            print(f"  Source failed: {exc}")

        should_checkpoint = (
            index % max(1, CHECKPOINT_EVERY_SOURCES) == 0
            or index == len(sources)
        )
        if should_checkpoint:
            successful_rows = [r for r in source_rows if r["source_id"] in finals_by_id]
            section_rows = make_section_rows(successful_rows, finals_by_id)
            combined_log = run_rows + API_CALL_LOG
            export_excel(source_rows, section_rows, definition_rows, fragment_rows, combined_log)
            print(f"  Checkpoint Excel saved: {RESULT_XLSX}")

    print(f"\nCompleted. Final workbook: {RESULT_XLSX}")
    return RESULT_XLSX


if RUN_ANALYSIS:
    result_path = run_pipeline()
    print(result_path)
else:
    print("RUN_ANALYSIS=False: no API calls will be made. Review the configuration and enable the flag to run.")





## Reproducibility notes

- The cache key includes source content and all analytical instructions. Changing either creates a new result version.
- API requests use store=False.
- The workbook retains source paths and fingerprints for traceability, while analytical prose omits author names.
- Definition quotes are exported only after a literal or whitespace-only normalised match against the source fragment.
- Official API documentation: https://developers.openai.com/api/reference/cli/resources/responses/methods/create
- Model availability depends on the OpenAI project. Change CHUNK_MODEL and SYNTHESIS_MODEL if necessary.
